# SHAP Analysis — Cohort Runner

Runs **SHAP Analysis** for all configured cohorts and age bands.

- **Script**: `7_shap_analysis/run_shap_analysis.py`
- **Cohorts**: `falls` (`fall_injury_any`), `ed` (`ed_event`)
- **Age bands**: **65–74 and 75–84** only
- **Outputs**: SHAP values (parquet), plots, feature importance rankings; synced to S3

In [ ]:
import os
import sys
from pathlib import Path

def resolve_python_bin() -> Path:
    env_bin = os.environ.get("COHORT_RUNNER_PYTHON")
    if env_bin:
        return Path(env_bin)
    return Path(sys.executable)

PYTHON_BIN = resolve_python_bin()
print(f"[INFO] Python binary: {PYTHON_BIN}")

def resolve_project_root() -> Path:
    if "__file__" in globals():
        root = Path(__file__).resolve().parent
        return root.parent if root.name == "7_shap_analysis" else root
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "7_shap_analysis" else cwd

PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_age_bands = ["65-74", "75-84"]
try:
    from py_helpers.constants import REQUIRED_COHORTS
    COHORT_NAMES = list(REQUIRED_COHORTS.keys())
    AGE_BANDS = _age_bands
except ImportError:
    COHORT_NAMES = ["falls", "ed"]
    AGE_BANDS = _age_bands

print(f"[INFO] Cohorts: {COHORT_NAMES}")
print(f"[INFO] Age bands (65–85 group): {AGE_BANDS}")

## Configuration

Select which cohorts and age bands to process. Leave empty lists to process all.

In [ ]:
COHORTS_TO_RUN = COHORT_NAMES   # ["falls", "ed"]
AGE_BANDS_TO_RUN = AGE_BANDS    # ["65-74", "75-84"]

print(f"Cohorts: {COHORTS_TO_RUN}")
print(f"Age bands: {AGE_BANDS_TO_RUN}")
print(f"Combinations: {len(COHORTS_TO_RUN) * len(AGE_BANDS_TO_RUN)}")

## Run SHAP Analysis

In [ ]:
import subprocess

FAIL_FAST = True

combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]
print(f"Running SHAP Analysis for {len(combinations)} combinations...")
print("=" * 60)

for cohort_name, age_band in combinations:
    print(f"\n[SHAP] {cohort_name} / {age_band}")
    result = subprocess.run(
        [str(PYTHON_BIN), "7_shap_analysis/run_shap_analysis.py",
         "--cohort", cohort_name,
         "--age_band", age_band],
        cwd=PROJECT_ROOT,
    )
    if result.returncode == 0:
        print(f"  COMPLETED: {cohort_name} / {age_band}")
    else:
        msg = f"  FAILED ({result.returncode}): {cohort_name} / {age_band}"
        print(msg)
        if FAIL_FAST:
            raise RuntimeError(msg)

print("\nAll SHAP Analysis runs complete.")